# 逐事件軌跡動畫(replay_epoch)

`example/replay_epoch.py` 用某個 epoch 存的權重、強制 `chunk_size=1` 重跑
`run_network_traced`,拿到逐事件精確的 `LayerForwardTrace`(`spike_mask`/
`v_steps`/`event_ms`,3 欄,見該檔案 docstring)。這份 notebook 把它依
`event_ms` 攤成真實時間軸播放成動畫:離散量(`spike_mask`)用瞬時顯示
(`viz.time_resample.resample_pulse`——spike 是瞬時事件,事件跟事件之間這個
neuron 根本沒被計算過,不是「還在 fire」,所以沒有事件的 frame 一律顯示沒
fire,不會沿用前一筆事件的值),連續量(`v_steps`)用衰減插值(`resample_decay`,
用該層的 `tau`,電位是真的連續存在的物理量,沿用前一筆事件的值再衰減才有
意義)——插值/動畫繪圖邏輯都在 `viz/`,這裡只負責讀資料、組裝、擺 UI。

**目前範圍只做 conv 層**(`out` 層是 FC,沒有 `unflatten_channels` 假設的
`oc/h/w` 空間結構,呈現方式留到之後再談)。

In [5]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import HTML, display

from example.paths import EXPERIMENTS_DIR
from example.replay_epoch import _load_train_sample, load_epoch_weights, replay_sample
from example.utils import load_run_record
from salt_core.layers import ConvLayer
from viz.channel_grid import ChannelGridAnimation, unflatten_channels
from viz.time_resample import build_frame_grid, resample_decay, resample_pulse


## 參數

改這裡,不改函式庫。

In [6]:
RUN_NAME = "conv_compressed_compressed_maxsteps_verify_20260913_120425"
EXP_DIR = str(EXPERIMENTS_DIR / RUN_NAME)
EPOCH = 0       # 要有對應的 weights/epoch_XXX.npz
DT_MS = 1.0     # 動畫的時間格解析度(毫秒),越小動畫越平滑但 frame 數越多
COMPARE_ROWS = 2   # 比較格子的列數(2x2 = 4 格)
COMPARE_COLS = 2   # 比較格子的欄數


## 讀這個 epoch 的層(幾何/`tau` 只跟 epoch 有關,跟 sample 無關,只需要讀一次)

`out` 層是 FC,先濾掉不放進選單。

In [7]:
LAYERS, _ = load_epoch_weights(EXP_DIR, EPOCH)
RUN_RECORD = load_run_record(EXP_DIR)

CONV_LAYERS = [l for l in LAYERS if isinstance(l, ConvLayer)]
CONV_LAYER_NAMES = [l.name for l in CONV_LAYERS]
print("可選的層:", CONV_LAYER_NAMES, "(out 層是 FC,之後再談,先不放進來)")


可選的層: ['conv1', 'conv2'] (out 層是 FC,之後再談,先不放進來)


## 組裝:一次比較 `COMPARE_ROWS x COMPARE_COLS` 組 (sample, layer, quantity, channel)

每一格獨立選 `sample`/`layer`/`quantity`/`channel`。全部格子共用**同一條真實 ms
時間軸**(取所有選到的 sample 裡最晚的真實事件時間當終點),塞進同一個
`ChannelGridAnimation`,用同一個 Play 按鈕真的同步播放,不是四個各自獨立、
要手動對頻道的播放器。

選到的層如果空間尺寸不一樣(例如 conv1 攤開 17×17、conv2 是 9×9),小的那格
會 padding 成跟最大的一樣大——padding 出來的格子畫成洋紅色
(`viz/channel_grid.py` 的 `_PAD_COLOR`),代表「這裡沒有神經元」,不會跟灰色
(沒 fire)或 viridis 深色端(連續值低值)搞混。

換 `sample`/`epoch` 才需要重跑 forward,所以 traces 用 `(epoch, sample)` 當
key cache 起來——只留最新一筆,不無限累積記憶體。

In [ ]:
_traces_cache = {}


def _get_traces(sample: int):
    key = (EPOCH, sample)
    if key not in _traces_cache:
        _traces_cache.clear()
        raw_sample = _load_train_sample(RUN_RECORD, sample)
        _traces_cache[key] = replay_sample(EXP_DIR, EPOCH, *raw_sample)
    return _traces_cache[key]


def _combo_frames(sample: int, layer_name: str, quantity: str, channel: int, frame_ms):
    """單一 (sample, layer, quantity, channel) 組合,套用共用的 frame_ms,回傳
    `(frames, is_discrete, title)`——`frames` 是這一格自己的 `(n_frames, H, W)`,
    還沒 padding。"""
    layer_idx = CONV_LAYER_NAMES.index(layer_name)
    layer = CONV_LAYERS[layer_idx]
    if not (0 <= channel < layer.oc):
        raise ValueError(f"{layer_name} channel 要在 [0, {layer.oc}) 之間,收到 {channel}")

    trace = _get_traces(sample)[layer_idx]
    event_ms = np.asarray(trace.event_ms)
    is_spike = quantity == "spike_mask"
    values = np.asarray(trace.spike_mask if is_spike else trace.v_steps)

    if is_spike:
        # spike 是瞬時事件,沒有事件的 frame 一律沒 fire,不沿用前一筆的值。
        resampled = resample_pulse(event_ms, values, frame_ms, dt=DT_MS, background=False)
    else:
        resampled = resample_decay(event_ms, values, frame_ms, tau=layer.tau, before_first=0.0)

    frames = np.stack([
        unflatten_channels(resampled[:, f], layer.oc, layer.h_out, layer.w_out)[channel]
        for f in range(resampled.shape[1])
    ]).astype(np.float64)  # (n_frames, H, W)
    title = f"s{sample} / {layer_name} / {quantity} / ch{channel}"
    return frames, is_spike, title


def _pad_to(frames, h_max: int, w_max: int):
    """把 (n_frames, h, w) 補到 (n_frames, h_max, w_max),多出來的邊界填 NaN
    (畫出來是洋紅色的 padding,見上面的說明 cell)。層的空間尺寸本來就一樣時
    這裡是 no-op。"""
    n_frames, h, w = frames.shape
    if h == h_max and w == w_max:
        return frames
    out = np.full((n_frames, h_max, w_max), np.nan)
    out[:, :h, :w] = frames
    return out


def show_comparison(combos: list):
    """`combos`:`[(sample, layer_name, quantity, channel), ...]`。全部塞進
    同一個 ChannelGridAnimation,同一條真實 ms 時間軸、同一個 Play 按鈕同步
    播放。"""
    t_end = max(
        float(np.nanmax(np.asarray(_get_traces(sample)[CONV_LAYER_NAMES.index(layer_name)].event_ms)))
        for sample, layer_name, _, _ in combos)
    frame_ms = build_frame_grid(0.0, t_end, DT_MS)

    per_combo = [_combo_frames(*combo, frame_ms) for combo in combos]
    h_max = max(f.shape[1] for f, _, _ in per_combo)
    w_max = max(f.shape[2] for f, _, _ in per_combo)
    frames = np.stack([_pad_to(f, h_max, w_max) for f, _, _ in per_combo], axis=1)  # (n_frames, n_combos, H, W)

    anim = ChannelGridAnimation(ncols=COMPARE_COLS).build(
        frames,
        titles=[title for _, _, title in per_combo],
        discrete=[is_spike for _, is_spike, _ in per_combo],
        frame_labels=[f"t = {ms:.1f} ms" for ms in frame_ms])
    plt.close(anim._fig)  # 避免 %matplotlib inline 把靜態圖跟 jshtml 動畫重複顯示
    display(HTML(anim.to_jshtml()))


def _combo_row(default_channel: int = 0):
    return widgets.HBox([
        widgets.IntText(value=0, description="sample", layout=widgets.Layout(width="150px")),
        widgets.Dropdown(options=CONV_LAYER_NAMES, description="layer", layout=widgets.Layout(width="190px")),
        widgets.Dropdown(options=["spike_mask", "v_steps"], description="quantity",
                          layout=widgets.Layout(width="220px")),
        widgets.IntText(value=default_channel, description="channel", layout=widgets.Layout(width="150px")),
    ])


_combo_rows = [_combo_row() for _ in range(COMPARE_ROWS * COMPARE_COLS)]
_compare_button = widgets.Button(description="比較")
_compare_output = widgets.Output()


def _on_compare_clicked(_):
    combos = [tuple(child.value for child in row.children) for row in _combo_rows]
    with _compare_output:
        _compare_output.clear_output(wait=True)
        show_comparison(combos)


_compare_button.on_click(_on_compare_clicked)
display(widgets.VBox(_combo_rows + [_compare_button, _compare_output]))
